# STCIR — Batch Experiment Runner

Runs the full pipeline on multiple datasets **sequentially** with automatic checkpointing.
You can interrupt at any time and re-run — completed phases are skipped.

**Datasets covered by default:**
- `mrtydi_arabic`  — Arabic Mr. TyDi  (prebuilt stage-1 + stage-2 if available)
- `mmarco_arabic`  — Arabic mMARCO    (prebuilt stage-1 if available)
- `msmarco`        — English MS MARCO (computed)

Prebuilt runs are detected automatically from the registry — no manual flags needed.
Annotation uses the Gemma4 LLM (human Flask UI is not available in batch mode).

In [ ]:
import subprocess, sys, os

# pip's startup calls os.getcwd(); if the kernel's cwd was deleted this raises
# FileNotFoundError. Recover the path from IPython's starting_dir (captured at
# kernel init, unaffected by later cwd changes or deletions).
try:
    _pkg = os.getcwd()
except FileNotFoundError:
    _ip  = get_ipython() if 'get_ipython' in dir() else None
    _pkg = getattr(_ip, 'starting_dir', None) or os.path.expanduser('~')
    os.chdir(_pkg)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", _pkg, "-q"], check=True)
print('✅ stcir installed')


## Cell 1 — Batch Configuration

In [ ]:
import os
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)s  %(message)s')

from stcir import STCIRConfig

# ═══════════════════════════════════════════════════════════════════════════
# HuggingFace token — required for private/gated prebuilt repos.
# Option A: set HF_TOKEN env var before launching Jupyter
# Option B: paste your token as a string below (never commit to git)
# ───────────────────────────────────────────────────────────────────────────
HF_TOKEN = os.getenv("HF_TOKEN", None)

# ═══════════════════════════════════════════════════════════════════════════
# BATCH DATASET LIST
# Each entry is loaded from a YAML config (edit configs/ to customise).
# annotation_mode is forced to 'llm' here — batch runs cannot use the Flask UI.
# ───────────────────────────────────────────────────────────────────────────
BATCH_CONFIGS = [
    STCIRConfig.from_yaml('configs/mrtydi_arabic.yaml'),
    STCIRConfig.from_yaml('configs/mmarco_arabic.yaml'),
    STCIRConfig.from_yaml('configs/msmarco_english.yaml'),
]

# Force LLM annotation for unattended runs
for cfg in BATCH_CONFIGS:
    cfg.annotation_mode = 'llm'

# ── Summary ──────────────────────────────────────────────────────────────
print(f'\n{"-"*60}')
print(f'  Batch queue: {len(BATCH_CONFIGS)} datasets')
print(f'  HF token   : {"set" if HF_TOKEN else "not set (public repos only)"}')
print(f'{"-"*60}')
for i, cfg in enumerate(BATCH_CONFIGS, 1):
    print(f'  [{i}] {cfg.dataset:<20}  lang={cfg.language:<8}'
          f'  stage1={cfg.stage1_source}  stage2={cfg.stage2_source}')


## Cell 2 — Run All Datasets

Each dataset runs all 5 phases end-to-end.  
**Checkpoints are per-dataset** — kill and restart anytime, only incomplete phases re-run.

In [ ]:
from stcir.pipeline.runner import PipelineRunner

all_results: dict = {}   # dataset → {config, results, qrels, ...}
failed:      list = []

for cfg in BATCH_CONFIGS:
    try:
        # auto_use_prebuilt=True: if PREBUILT_FOLDER_MAP has an entry for this
        # dataset, stage1_source / stage2_source are overridden automatically.
        # hf_token is forwarded to all prebuilt download calls.
        runner = PipelineRunner(cfg, auto_use_prebuilt=True, hf_token=HF_TOKEN)
        result = runner.run_all()
        all_results[cfg.dataset] = result
    except Exception as exc:
        import traceback
        print(f'\n❌  {cfg.dataset} FAILED: {exc}')
        traceback.print_exc()
        failed.append((cfg.dataset, str(exc)))

print(f'\n{"="*60}')
print(f'  Batch complete: {len(all_results)} succeeded, {len(failed)} failed')
if failed:
    for ds, err in failed:
        print(f'  ✗ {ds}: {err}')
print(f'{"="*60}')


## Cell 3 — Per-Dataset Results Summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

for dataset, res in all_results.items():
    df = res['results']
    print(f'\n── {dataset} ─────────────────────────────')
    display(df.style.highlight_max(axis=0, color='#c8e6c9').format('{:.4f}'))

## Cell 4 — Cross-Dataset Comparison (Full-Pipeline nDCG@10)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

COMPARE_METRIC = 'nDCG@10'    # change to any metric in config.metrics
COMPARE_SYSTEM = 'Full-Pipeline'

rows = []
for dataset, res in all_results.items():
    df = res['results']
    if COMPARE_SYSTEM in df.index and COMPARE_METRIC in df.columns:
        rows.append({'dataset': dataset, COMPARE_METRIC: df.loc[COMPARE_SYSTEM, COMPARE_METRIC]})

if rows:
    summary = pd.DataFrame(rows).set_index('dataset')
    display(summary.style.highlight_max(color='#c8e6c9').format('{:.4f}'))

    fig, ax = plt.subplots(figsize=(7, 4))
    summary[COMPARE_METRIC].plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'{COMPARE_SYSTEM} — {COMPARE_METRIC} across datasets')
    ax.set_ylabel(COMPARE_METRIC)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig('outputs/batch_comparison.png', dpi=150)
    plt.show()
else:
    print('No results to compare yet.')

## Cell 5 — Cross-Dataset System Correlation (Kendall's τ · Spearman's ρ)

In [ ]:
from stcir.topics.loader import load_reference_qrels
from stcir.evaluation.correlation import compute_system_correlation, global_rank_correlation

CORR_DATASETS = ['mrtydi_arabic', 'mmarco_arabic', 'msmarco']

for dataset in CORR_DATASETS:
    if dataset not in all_results:
        continue
    res = all_results[dataset]
    try:
        print(f'\n── {dataset}: loading reference qrels …')
        human_qrels = load_reference_qrels(dataset)
    except Exception as e:
        print(f'   Skipping correlation for {dataset}: {e}')
        continue

    corr_df = compute_system_correlation(
        human_qrels     = human_qrels,
        synthetic_qrels = res['qrels'],
        runs            = res.get('system_runs', {}),
        metrics         = ['MRR@10', 'nDCG@10', 'Recall@10'],
    )
    display(corr_df.style.format('{:.4f}'))

    import tempfile, os
    from stcir.utils import save_qrels
    tmp = tempfile.NamedTemporaryFile(suffix='.tsv', delete=False); tmp.close()
    save_qrels(human_qrels,      tmp.name + '.human')
    save_qrels(res['qrels'],     tmp.name + '.synth')
    gc = global_rank_correlation(
        human_qrels_path     = tmp.name + '.human',
        synthetic_qrels_path = tmp.name + '.synth',
    )
    os.unlink(tmp.name + '.human'); os.unlink(tmp.name + '.synth')
    print(f'   Global Kendall\'s τ  : {gc["global_kendall_tau"]:.4f}')
    print(f'   Global Spearman\'s ρ : {gc["global_spearman_rho"]:.4f}')

## Cell 6 — Publish All Datasets to HuggingFace (`hatemestinbejaia/STCALIR-Dataset`)

In [ ]:
import os
from stcir.publishing.hub import push_batch_to_hub, STCALIR_DATASET_REPO

assert all_results, "Run Cell 2 first (no results to publish yet)"

# ── HuggingFace credentials ───────────────────────────────────────────────
# Option A: set HF_TOKEN env var before launching Jupyter
# Option B: run `huggingface-cli login` in a terminal beforehand
HF_TOKEN = os.getenv("HF_TOKEN", None)

# ── Push all datasets in one DatasetDict ─────────────────────────────────
# Each dataset becomes three splits named <dataset>_collection,
# <dataset>_topics, and <dataset>_qrels.
# Example splits for a 3-dataset batch:
#   mrtydi_arabic_collection, mrtydi_arabic_topics, mrtydi_arabic_qrels
#   mmarco_arabic_collection,  mmarco_arabic_topics,  mmarco_arabic_qrels
#   msmarco_collection,        msmarco_topics,        msmarco_qrels
url = push_batch_to_hub(
    batch_results = all_results,
    repo_id       = STCALIR_DATASET_REPO,   # hatemestinbejaia/STCALIR-Dataset
    private       = False,
    token         = HF_TOKEN,
)

print(f"\n✅ All datasets published → {url}")
